# Denoising and Debluring with TpV Chambolle-Pock
This notebook contains an implementation of the Chambolle-Pock algorithm with TpV regularization.
In addition, this notebook includes four sections dedicated to testing the CP algorithm and evaluating different values of the parameter p and the regularization parameter.

## Libraries & Drive Access & some constants

In [ ]:
!pip install astra-toolbox

import os
import sys
import time
import random
import shutil
import numpy as np
import torch
import torchvision
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Any
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from google.colab import drive
from PIL import Image

# for Data & Image check
import json
import math
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datasets import load_from_disk, load_dataset_builder
from collections import Counter

drive.mount('/content/drive')
os.environ["CACHE"] = "/content/cache"

BASE_DIR = Path("/content/drive/MyDrive/Computational Imaging")
sys.path.append(str(BASE_DIR))

from IPPy.IPPy.operators import Blurring
import IPPy.IPPy.utilities as IPPy_utils
from IPPy.IPPy.operators import Gradient

# Constants
DEVICE = IPPy_utils.get_device()

## Basic functions and configuration
Configurations of parameters like noise, type of blur, dataset and samples per class

In [ ]:
@dataclass(frozen=True)
class Config:
  seed: int = 42
  dataset_name: str = "benjamin-paine/imagenet-1k-256x256"
  target_classes: List[int] = (0, 10, 100, 200) # default_factory with lambda to have each istance of Config it's own target_classes
  samples_per_class: int = 1300 #max

  drive_output_dir: Path = BASE_DIR / "dataset"

  # Forward operator parameters
  img_shape: Tuple[int, int, int] = (3, 256, 256) # img type, (C,H,W)
  blur_kernel_type: str = "gaussian"
  blur_kernel_size: int = 9
  blur_sigma: float = 2.0
  motion_angle: float = 45.0 # unused rn

  noise_levels: List[float] = (0.005, 0.01, 0.05, 0.1)
  device: str = DEVICE

def set_seed(seed: int = 42) -> None:
  """ Set seed for reproducibility """
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False
  torch.use_deterministic_algorithms(True, warn_only=True)

# Image transformation
pil_to_tensor = torchvision.transforms.ToTensor()
tensor_to_pil = torchvision.transforms.ToPILImage()

## Data Loading and Check

In this cell the dateset is loaded from drive.
In addition, this cell includes a feature that monitors the level of noise and blur in the images.

In [ ]:
config = Config()
set_seed(config.seed)

blur = Blurring(
  img_shape=config.img_shape,
  kernel_type=config.blur_kernel_type,
  kernel_size=config.blur_kernel_size,
  kernel_variance=config.blur_sigma**2
)

final_dataset = load_from_disk(str(config.drive_output_dir))
print(f"Dataset caricato da: {config.drive_output_dir}")


# To keep the environment state perfectly consistent, we merge the various splits of `final_dataset` to recreate the `ds_degraded` variable
from datasets import concatenate_datasets
ds_degraded = concatenate_datasets([
    final_dataset['train'],
    final_dataset['validation'],
    final_dataset['test']
])


# Let's read the dataset's metadata so that these variables are always available in the environment
builder = load_dataset_builder(config.dataset_name)
class_names = builder.info.features["label"].names

# for label of the class used
if (config.drive_output_dir / "labels.json").exists():
  # reading from the file on drive
  with open(config.drive_output_dir / "labels.json") as f:
    id2label = json.load(f)
    id2label = {int(k): v for k, v in id2label.items()}
else:
  # reading from the official dataset
  id2label = {cls: class_names[cls] for cls in config.target_classes}

print("=== Split sizes ===")
for split, ds_split in final_dataset.items():
  print(f"  {split:>10}: {len(ds_split):>5} samples")

total = sum(len(v) for v in final_dataset.values())
print(f"  {'TOTAL':>10}: {total:>5} samples")

print("\n=== Label distribution ===")
for split, ds_split in final_dataset.items():
  counts = Counter(ds_split['label'])
  print(f"  {split}: { {k: counts[k] for k in sorted(counts)} }")

print("\n=== Columns ===")
print(" ", final_dataset['train'].column_names)

print("\n=== Pixel value ranges (first 5 train samples) ===")
sample = final_dataset['train'].select(range(5))
for col in ['x'] + [f"y_{int(round(sigma_n * 1000)):03d}" for sigma_n in config.noise_levels]:
  vals = [np.array(img) for img in sample[col]]
  arr = np.stack(vals)
  print(f"  {col}: min={arr.min()}, max={arr.max()}, dtype={arr.dtype}")

#  verify ||e|| / ||Kx|| ≈ sigma_n for each level of noise
print("\n=== Noise level verification (mean over 20 samples) ===")
sample = final_dataset['train'].select(range(20))

x_tensors   = torch.stack([pil_to_tensor(img) for img in sample['x']]).float().to(config.device)
y_clean_est = blur(x_tensors)  # Kx

for sigma_n in config.noise_levels:
  key = f"y_{int(round(sigma_n * 1000)):03d}"
  y_tensors = torch.stack([pil_to_tensor(img) for img in sample[key]]).float().to(config.device)
  noise     = y_tensors - y_clean_est
  measured_sigma = noise.std(dim=(1, 2, 3))
  print(f"  {key}: expected={sigma_n:.3f}  measured={measured_sigma.mean():.6f} ± {measured_sigma.std():.6f}")

print("\n=== Blur MSE ===")
mse = ((x_tensors - y_clean_est) ** 2).mean()
print("Blur MSE:", mse.item())

print("\n=== Image Shape ===")
sample = final_dataset["train"][0]

for key in ["x"] + [
    f"y_{int(round(s*1000)):03d}"
    for s in config.noise_levels
]:
    print(key, np.array(sample[key]).shape)

# === Plotting ===
cols_to_show = ['x'] + [f"y_{int(round(s*1000)):03d}" for s in config.noise_levels]
titles       = ['x (clean)', *[f"y σ={s}" for s in config.noise_levels]]
n_per_class  = 2

n_rows = len(config.target_classes) * n_per_class
n_cols = len(cols_to_show)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(n_cols * 3, n_rows * 3)
)

labels_array = np.array(final_dataset['train']['label'])

row = 0
for cls in config.target_classes:
    all_indices = np.where(labels_array == cls)[0][:n_per_class].tolist()

    for idx in all_indices:
        sample_row = final_dataset['train'][int(idx)]
        for col_i, col in enumerate(cols_to_show):
            ax = axes[row][col_i]
            ax.imshow(sample_row[col])
            ax.axis('off')
            if col_i == 0:
                ax.set_ylabel(f"{id2label[cls]}", fontsize=9, rotation=90, labelpad=4)
            if row == 0:
                ax.set_title(titles[col_i], fontsize=9)
        row += 1

plt.suptitle("Degradation check: x vs y per noise level", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(str(config.drive_output_dir / "image_check.png"), dpi=100, bbox_inches='tight')
plt.show()

## PSNR and SSIM Function

In this cell is defined the function that computes the PSNR and SSIM metrics given the ground truth and the predicted image.

In [ ]:
def compute_metrics(x_pred, x_true):
    """
    Computes PSNR and SSIM metrics between the predicted and ground truth PyTorch tensors
    (assuming a batch size of B=1).
    """

    # Move the tensors from GPU to CPU, detach them from the computational graph,
    # and convert them to NumPy format (Height, Width, Channels) required by scikit-image
    img_pred = x_pred[0].cpu().detach().permute(1, 2, 0).numpy()
    img_true = x_true[0].cpu().detach().permute(1, 2, 0).numpy()

    # The image pixel values are in the range [0, 1] due to the initial tensor normalization
    psnr_val = psnr_metric(img_true, img_pred, data_range=1.0)

    # channel_axis=-1 indicates that the RGB channels are located in the last dimension
    ssim_val = ssim_metric(img_true, img_pred, data_range=1.0, channel_axis=-1)

    return psnr_val, ssim_val

## TpV Function Definition with Adam (Optional)

This cell contains the definition of the tpv method for solving the deblur and denoise problem using the Adam optimizer.

In [ ]:
# @title
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric


def total_p_variation_loss(img, p=1.0):
    """ Computes the Total p-Variation regularization loss of the image """

    # Compute the absolute differences between adjacent pixels along the vertical axis (height)
    diff_h = torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :])
    # Compute the absolute differences between adjacent pixels along the horizontal axis (width)
    diff_w = torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1])

    # Add a small epsilon (1e-8) to prevent division by zero or NaN gradients (numerical stability)
    # Raise the differences to the power of 'p' and sum all the values
    tv_h = torch.pow(diff_h + 1e-8, p).sum()
    tv_w = torch.pow(diff_w + 1e-8, p).sum()

    # Return the total p-variation loss combining both vertical and horizontal components
    return tv_h + tv_w

def deblur_denoise_tpv(y, K, x_clean=None, lambda_tpv=0.01, p=1.0, num_iters=500, lr=0.01, device='cuda'):
    """
    Solves the inverse problem (deblurring and denoising) using Total p-Variation
    and tracks performance metrics over iterations.
    """

    # Initialize the target image 'x' as a clone of the degraded image 'y' (initial guess)
    x = y.clone().detach().to(device)

    # Enable gradient tracking for 'x', as this is the variable we want to optimize/update
    x.requires_grad_(True)

    # Initialize the Adam optimizer to update only the tensor 'x' with the given learning rate
    optimizer = torch.optim.Adam([x], lr=lr)

    # Dictionary for storing the "history" of the optimization (losses and metrics over time)
    history = {'loss': [], 'psnr': [], 'ssim': [], 'iters': []}

    # Main optimization loop
    for i in range(num_iters):
        # Reset the gradients of the optimizer to zero before each backward pass
        optimizer.zero_grad()

        # FORWARD PASS
        # Apply the degradation operator K (blur) to our current estimate 'x'
        y_hat = K(x)

        # Compute the Data Fidelity term: Mean Squared Error between the blurred estimate and the observed noisy image
        data_loss = 0.5 * F.mse_loss(y_hat, y, reduction='sum')

        # Compute the Regularization term: Total p-Variation loss
        tv_loss = total_p_variation_loss(x, p=p)

        # Combine both terms to get the total objective loss to minimize
        loss = data_loss + lambda_tpv * tv_loss

        # BACKWARD PASS
        # Compute the gradients of the loss with respect to all pixels in 'x'
        loss.backward()

        # Update the pixel values of 'x' based on the computed gradients
        optimizer.step()

        # Clip the pixel values of 'x' to ensure they remain in the valid image range [0, 1]
        # This operation is wrapped in torch.no_grad() because it shouldn't affect gradient tracking
        with torch.no_grad():
            x.clamp_(0, 1)

        # Track the total loss at every iteration
        history['loss'].append(loss.item())

        # Calculate and track visual metrics at regular intervals (every 10% of iterations) and at the last step
        if (i + 1) % (num_iters // 10) == 0 or i == num_iters - 1:
            if x_clean is not None:
                # If the ground truth image is available, compute PSNR and SSIM
                psnr_val, ssim_val = compute_metrics(x, x_clean)

                # Store the current iteration number and the computed metrics
                history['iters'].append(i + 1)
                history['psnr'].append(psnr_val)
                history['ssim'].append(ssim_val)
            else:
                # If ground truth is not provided, just print the current iteration and loss
                print(f"Iter {(i + 1):04d}/{num_iters} | Loss: {loss.item():.6f}")

    # Return the final optimized image (detached from the graph) and the optimization history
    return x.detach(), history


## TpV with Chambolle-Pock Function Definition
This cell contains the implementation of the Chambolle-Pock algorithm with TpV for solving the deblur and denoise problem.

In [ ]:

def deblur_denoise_tpv_chambolle_pock(
    noisy_blurred_image,
    blur_operator,
    ground_truth_image=None,
    tv_weight=0.01,
    p_norm_value=0.5,
    num_iters=300,
    device='cuda'
):
    """
    Solves the Deblurring and Denoising problem using Total p-Variation (supports p <= 1).
    It implements the Primal-Dual Hybrid Gradient (Chambolle-Pock) algorithm
    with Iteratively Reweighted L1 (IR-PDHG) to handle the non-convexity of p < 1.
    """
    # =========================================================================
    # PRIMAL VARIABLES INITIALIZATION (Image Space)
    # =========================================================================

    # current_image: The actual image we are trying to reconstruct.
    # Initialized with the noisy/blurred observation.
    current_image = noisy_blurred_image.clone().detach().to(device)

    # extrapolated_image: A "dummy" image, pushed forward by calculating the
    # momentum of the last step. It is required by the algorithm to guarantee convergence.
    extrapolated_image = current_image.clone().detach().to(device)
    # =========================================================================
    # DUAL VARIABLES INITIALIZATION (Constraint/Gradient Space)
    # =========================================================================

    # dual_data_fidelity: Dual variable that handles the error between the image
    # we are building and the original noisy image we observed.
    dual_data_fidelity = torch.zeros_like(noisy_blurred_image).to(device)

    # dual_tv_horizontal / dual_tv_vertical: Dual variables that track the energy
    # of the spatial gradient (differences between adjacent pixels).
    # They are used to enforce the Total Variation regularization.
    dual_tv_horizontal = torch.zeros_like(current_image).to(device)
    dual_tv_vertical = torch.zeros_like(current_image).to(device)
    # =========================================================================
    # ALGORITHM HYPERPARAMETERS
    # =========================================================================

    # primal_step_size (tau): The learning rate/step size for updating the image pixels.
    primal_step_size = 0.3

    # dual_step_size (sigma): The learning rate/step size for updating the dual variables.
    dual_step_size = 0.3

    # extrapolation_factor (theta): The amount of "momentum" to give to the extrapolated_image.
    # It must strictly be 1.0 to preserve the mathematical convergence theorems.
    extrapolation_factor = 1.0

    # smoothing_epsilon (eta): A tiny constant to prevent division by zero
    # during the reweighting step when there are perfectly flat areas (es solid background).
    smoothing_epsilon = 2e-3
    history = {'loss': [], 'psnr': [], 'ssim': [], 'iters': []}
    for i in range(num_iters):
        # =========================================================
        # PHASE 0: REWEIGHTING FACTOR COMPUTATION (For p < 1)
        # =========================================================
        with torch.no_grad():
            # Calculate the color difference between pixels (the spatial gradient of the current image)
            grad_horizontal = torch.zeros_like(current_image)
            grad_vertical = torch.zeros_like(current_image)
            grad_horizontal[:, :, :, :-1] = current_image[:, :, :, 1:] - current_image[:, :, :, :-1]
            grad_vertical[:, :, :-1, :] = current_image[:, :, 1:, :] - current_image[:, :, :-1, :]
            # The squared mathematical magnitude (length) of the gradient vector
            spatial_grad_mag_sq = grad_horizontal**2 + grad_vertical**2

            # reweighting_factor (W): The Weight matrix. It translates our non-convex
            # problem into a temporarily convex one, allowing Chambolle-Pock to work safely.
            reweighting_factor = (torch.sqrt(smoothing_epsilon**2 + spatial_grad_mag_sq) / smoothing_epsilon) ** (p_norm_value - 1.0)
        # =========================================================
        # PHASE 1: DUAL VARIABLES UPDATE (Dual Ascent)
        # =========================================================
        with torch.no_grad():
            # --- Data Fidelity Dual Update ---

            # Blur the extrapolated image so we can compare it with the noisy observation
            blurred_extrapolated = blur_operator(extrapolated_image)

            # dual_data_fidelity_new: Explicitly computes the Proximal Operator for the L2 norm (Squared Error)
            # Proximal Operator for the L2 norm is a vector shrinkage operation. It scales the input vector v towards the origin, reducing its magnitude by λ while preserving its direction
            # Proximality operators are used to replace classic gradient descent in order to handle non differential functions
            # They find the point that reduces the function while remaining near the original point
            # q_new = (q_old + sigma*(Kx_pred - y))/1+sigma
            dual_data_fidelity_new = (dual_data_fidelity + (dual_step_size * blurred_extrapolated - dual_step_size * noisy_blurred_image)) / (1.0 + dual_step_size)

            # --- Total Variation Dual Update ---

            # Compute the spatial gradient of the extrapolated image
            grad_extrap_h = torch.zeros_like(extrapolated_image)
            grad_extrap_v = torch.zeros_like(extrapolated_image)
            grad_extrap_h[:, :, :, :-1] = extrapolated_image[:, :, :, 1:] - extrapolated_image[:, :, :, :-1]
            grad_extrap_v[:, :, :-1, :] = extrapolated_image[:, :, 1:, :] - extrapolated_image[:, :, :-1, :]

            # ascent_tv: Gradient Ascent step along the direction provided by the image gradient
            ascent_tv_h = dual_tv_horizontal + dual_step_size * grad_extrap_h
            ascent_tv_v = dual_tv_vertical + dual_step_size * grad_extrap_v

            # dual_tv_magnitude: The length of the newly ascended TV dual vectors
            dual_tv_magnitude = torch.sqrt(ascent_tv_h**2 + ascent_tv_v**2)

            # Proximal operator is the geometric projection n the L-infinity ball
            # projection_radius: The maximum radius allowed by the lambda penalty and our Reweighting factor
            projection_radius = tv_weight * reweighting_factor
            # projection_scale: If the vector exceeded the radius, it is scaled down (clamped) to stay within the limit.
            projection_scale = torch.clamp(projection_radius / (dual_tv_magnitude + 1e-8), max=1.0)

            # dual_tv_new: The new Dual values, geometrically clipped if necessary.
            dual_tv_horizontal_new = ascent_tv_h * projection_scale
            dual_tv_vertical_new = ascent_tv_v * projection_scale

            # --- Commit the dual updates ---
            dual_data_fidelity = dual_data_fidelity_new
            dual_tv_horizontal = dual_tv_horizontal_new
            dual_tv_vertical = dual_tv_vertical_new

        #=========================================================
        # PHASE 2: PRIMAL VARIABLE UPDATE (Primal Descent)
        # =========================================================

        # We temporarily re-enable PyTorch's Autograd to use the Vector-Jacobian Product (VJP) trick.
        # Chambolle-Pock requires computing: x_new = x_old - tau * (K^T * q1 + grad^T * q2)
        # Writing transposed operators (K^T and grad^T) manually is complex and error-prone.
        image_temp = current_image.clone().requires_grad_(True)
        blurred_temp = blur_operator(image_temp)
        grad_temp_h = torch.zeros_like(image_temp)
        grad_temp_v = torch.zeros_like(image_temp)
        grad_temp_h[:, :, :, :-1] = image_temp[:, :, :, 1:] - image_temp[:, :, :, :-1]
        grad_temp_v[:, :, :-1, :] = image_temp[:, :, 1:, :] - image_temp[:, :, :-1, :]
        # We construct a dummy scalar S = <Kx, q1> + <grad_x, q2>.
        # By linear algebra definition, <Ax, q> = <x, A^T * q>, so S is mathematically equivalent to:
        # S = x * (K^T * q1 + grad^T * q2)
        inner_product = torch.sum(blurred_temp * dual_data_fidelity) + torch.sum(grad_temp_h * dual_tv_horizontal) + torch.sum(grad_temp_v * dual_tv_vertical)

        # When we call .backward(), PyTorch computes the derivative of S with respect to x.
        # Since S = x * C, the derivative is exactly C = (K^T * q1 + grad^T * q2)
        # This extracts the exact transposed operations we need for free, without programming them manually.
        inner_product.backward()
        # adjoint_gradients_sum contains the perfectly extracted (K^T * q1 + grad^T * q2)
        adjoint_gradients_sum = image_temp.grad
        with torch.no_grad():
            # updated_image: Perform the Primal Gradient Descent step
            updated_image = current_image - primal_step_size * adjoint_gradients_sum

            # Protection: Force the physical image pixels to remain within the valid visibility range [0.0 (Black), 1.0 (White)]
            updated_image = torch.clamp(updated_image, 0.0, 1.0)
             # =========================================================
            # PHASE 3: EXTRAPOLATION (Acceleration Step)
            # =========================================================
            # In a Primal-Dual problem, we are searching for a saddle point (min-max).
            # Without momentum, the variables would spiral endlessly around the saddle center.
            # We "throw the image forward" by repeating the exact step we just took:
            # x_bar = x_new + 1.0 * (x_new - x_old) = 2*x_new - x_old
            # Passing this future prediction to the Dual variable prevents infinite oscillations
            # and mathematically guarantees the optimal O(1/N) convergence rate.
            extrapolated_image = updated_image + extrapolation_factor * (updated_image - current_image)

            # Update the actual primal image for the next cycle
            current_image = updated_image.clone()
        # =========================================================
        # METRICS AND LOGGING
        # =========================================================
        if (i + 1) % (num_iters // 10) == 0 or i == num_iters - 1:
            with torch.no_grad():
                reconstructed_blur = blur_operator(current_image)
                data_loss = 0.5 * torch.sum((reconstructed_blur - noisy_blurred_image) ** 2)
                grad_h = torch.zeros_like(current_image)
                grad_v = torch.zeros_like(current_image)
                grad_h[:, :, :, :-1] = current_image[:, :, :, 1:] - current_image[:, :, :, :-1]
                grad_v[:, :, :-1, :] = current_image[:, :, 1:, :] - current_image[:, :, :-1, :]
                grad_mag = torch.sqrt(grad_h**2 + grad_v**2 + 1e-8)
                tv_loss = torch.sum(torch.pow(grad_mag, p_norm_value))
                loss = data_loss + tv_weight * tv_loss
                history['loss'].append(loss.item())
                if ground_truth_image is not None:
                    # NOTE: assuming `compute_metrics` is imported/defined globally
                    psnr_val, ssim_val = compute_metrics(current_image, ground_truth_image)
                    history['iters'].append(i + 1)
                    history['psnr'].append(psnr_val)
                    history['ssim'].append(ssim_val)
    return current_image.detach(), history

## Noise Level Setting and Image Retrival

In this cell we set the desired noise level; also we retrive and transform the image we want to evaluete and clean with TpV

In [ ]:
#   Noise lavel that we want to use
noise_key = 'y_010'
#   Set the current set of images used as the test set
ds_degraded = final_dataset['test']
#   Retrieve 1 image (e.g., sample 1) and create the tensors
sample = ds_degraded[23]
#   Noisy Tensor
y_noisy = pil_to_tensor(sample[noise_key]).unsqueeze(0).to(DEVICE)
#   Clean Tensor
x_clean = pil_to_tensor(sample['x']).unsqueeze(0).to(DEVICE)

## Deblur and Denoise with TpV Test

In this cell we test the TpV function on the image selected in the previous cell and visualize the results

In [ ]:
print("Starting TpV Optimization...\n")

# Run the deblurring and denoising algorithm
# The optimization will update 'x' for 300 iterations to minimize the loss function
x_reconstructed, history = deblur_denoise_tpv_chambolle_pock(
      y=y_noisy,          # The degraded (blurred and noisy) input image
      K=blur,             # The degradation operator (blur kernel)
      x_clean=x_clean,    # Ground truth image (used ONLY for calculating metrics, not for optimization)
      lambda_tpv= 0.0005, # Regularization weight (how strongly to penalize noise/gradients)
      p=0.5,              # Power parameter for the non-convex Total p-Variation
      num_iters=500,      # Number of optimization steps
      lr=0.02,            # Learning rate for the Adam optimizer
      device=DEVICE       # Hardware accelerator (CPU or CUDA)
  )


# PLOTTING THE IMAGES (VISUAL COMPARISON)

# Convert the PyTorch tensors back to NumPy arrays (Height, Width, Channels) for visualization
img_clean_np = x_clean[0].cpu().permute(1, 2, 0).numpy()
img_noisy_np = y_noisy[0].cpu().permute(1, 2, 0).numpy()
img_recon_np = x_reconstructed[0].cpu().permute(1, 2, 0).numpy()

# Calculate the baseline metrics comparing the degraded image with the ground truth
psnr_start, ssim_start = compute_metrics(y_noisy, x_clean)

# Create a figure with 3 subplots side-by-side
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Subplot 1: The original, untouched clean image
axs[0].imshow(img_clean_np)
axs[0].set_title('Original (Clean)')
axs[0].axis('off')

# Subplot 2: The degraded input image with its baseline metrics
axs[1].imshow(img_noisy_np)
axs[1].set_title(f"Degraded\nPSNR: {psnr_start:.2f} dB | SSIM: {ssim_start:.6f}")
axs[1].axis('off')

# Subplot 3: The final reconstructed image with its final metrics achieved at the last iteration
axs[2].imshow(img_recon_np)
axs[2].set_title(f"Reconstructed (TpV)\nPSNR: {history['psnr'][-1]:.2f} dB | SSIM: {history['ssim'][-1]:.6f}")
axs[2].axis('off')

# Adjust layout to prevent overlap and display the figure
plt.tight_layout()
plt.show()


# PLOTTING THE OPTIMIZATION METRICS EVOLUTION

# Create a second figure with 3 subplots to track the learning curves
fig, axs = plt.subplots(1, 3, figsize=(16, 4))

# Subplot 1: Total Loss Evolution
# Shows how the objective function (Data Fidelity + TpV Regularization) decreases over iterations
axs[0].plot(history['loss'], color='red')
axs[0].set_title('Total Loss per Iteration')
axs[0].set_xlabel('Iterations')
axs[0].set_ylabel('Loss')
axs[0].grid(True)

# Subplot 2: PSNR Evolution
# Shows how the PSMR improves compared to the baseline
axs[1].plot(history['iters'], history['psnr'], marker='o', color='blue')
axs[1].axhline(y=psnr_start, color='gray', linestyle='--', label='Degraded Baseline')
axs[1].set_title('PSNR Evolution')
axs[1].set_xlabel('Iterations')
axs[1].set_ylabel('PSNR [dB]')
axs[1].legend()
axs[1].grid(True)

# Subplot 3: SSIM Evolution
# Shows how the SSIM improves compared to the baseline
axs[2].plot(history['iters'], history['ssim'], marker='o', color='green')
axs[2].axhline(y=ssim_start, color='gray', linestyle='--', label='Degraded Baseline')
axs[2].set_title('SSIM Evolution')
axs[2].set_xlabel('Iterations')
axs[2].set_ylabel('SSIM [0-1]')
axs[2].legend()
axs[2].grid(True)

# Adjust layout and display the metrics figure
plt.tight_layout()
plt.show()


## 2D "p" Evaluation Over One Image
In this cell we evaluate the SSIM and PSNR metrics of one image with "lmbda_tpv" fixed to find the best value of the parameter "p"

In [ ]:
# @title
import matplotlib.cm as cm

# Calculate the baseline starting metrics between the degraded image and the ground truth
psnr_start, ssim_start = compute_metrics(y_noisy, x_clean)

# Generate an array of 10 'p' values evenly spaced between 0.1 and 0.5 (step size 0.05)
p_values = np.linspace(0.1, 0.5, 10)

# We use the round function to avoid Python's floating-point precision artifacts
for i in range(len(p_values)):
  p_values[i] = round(p_values[i], 2)

# Dictionary to store the results of the optimization for each tested 'p' value
results_p = {}

print(f"Starting Tuning for parameter 'p' on the image with noise level {noise_key}...\n")

for p_val in p_values:
    print(f"--- Optimization with p = {p_val:.2f} ---")

    # Run the TpV algorithm keeping all other hyperparameters constant
    x_reconstructed, history = deblur_denoise_tpv(
        y=y_noisy,
        K=blur,
        x_clean=x_clean,
        lambda_tpv=0.0005,  # Keep the regularization weight constant
        p=p_val,           # Vary the 'p' parameter
        num_iters=500,     # Number of iterations for tuning
        lr=0.005,
        device=DEVICE
    )

    # Store the reconstructed image and its learning history for the current 'p'
    results_p[p_val] = {
        'x_reconstructed': x_reconstructed,
        'history': history
    }


# PLOT 1: METRICS EVOLUTION OVER ITERATIONS

fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Use a continuous colormap to smoothly color the lines representing different 'p' values
colors = cm.viridis(np.linspace(0, 0.9, len(p_values)))

for i, p_val in enumerate(p_values):
    history = results_p[p_val]['history']
    c = colors[i]
    label = f'p = {p_val:.2f}'

    # To avoid cluttering the graphs, we use continuous lines without scatter markers
    axs[0].plot(history['loss'], color=c, label=label)
    axs[1].plot(history['iters'], history['psnr'], color=c, label=label)
    axs[2].plot(history['iters'], history['ssim'], color=c, label=label)

# Setup subplots details (titles, labels, legends, grids)
axs[0].set_title('Total Loss Evolution')
axs[0].set_xlabel('Iterations')
axs[0].set_ylabel('Loss')
axs[0].legend(fontsize='small')
axs[0].grid(True)

axs[1].axhline(y=psnr_start, color='gray', linestyle='--', label='Baseline')
axs[1].set_title('PSNR Evolution')
axs[1].set_xlabel('Iterations')
axs[1].set_ylabel('PSNR [dB]')
axs[1].legend(fontsize='small')
axs[1].grid(True)

axs[2].axhline(y=ssim_start, color='gray', linestyle='--', label='Baseline')
axs[2].set_title('SSIM Evolution')
axs[2].set_xlabel('Iterations')
axs[2].set_ylabel('SSIM [0-1]')
axs[2].legend(fontsize='small')
axs[2].grid(True)

plt.tight_layout()
plt.show()


# PLOT 2: FINAL SUMMARY GRAPHS

# Extract the final metric value reached at the last iteration for each tested 'p'
final_psnr = [results_p[p_val]['history']['psnr'][-1] for p_val in p_values]
final_ssim = [results_p[p_val]['history']['ssim'][-1] for p_val in p_values]

fig, axs = plt.subplots(1, 2, figsize=(15, 5))

# Plot Final PSNR vs 'p' value
axs[0].plot(p_values, final_psnr, marker='o', markersize=8, color='dodgerblue', linewidth=2)
axs[0].axhline(y=psnr_start, color='gray', linestyle='--', label='Degraded Baseline')
axs[0].set_title(f'Final PSNR vs "p" (Noise: {noise_key})', fontsize=14)
axs[0].set_xlabel('Parameter "p" Value', fontsize=12)
axs[0].set_ylabel('Final PSNR [dB]', fontsize=12)
axs[0].grid(True, linestyle=':')
axs[0].set_xticks(p_values)
axs[0].legend()

# Plot Final SSIM vs 'p' value
axs[1].plot(p_values, final_ssim, marker='s', markersize=8, color='forestgreen', linewidth=2)
axs[1].axhline(y=ssim_start, color='gray', linestyle='--', label='Degraded Baseline')
axs[1].set_title(f'Final SSIM vs "p" (Noise: {noise_key})', fontsize=14)
axs[1].set_xlabel('Parameter "p" Value', fontsize=12)
axs[1].set_ylabel('Final SSIM [0-1]', fontsize=12)
axs[1].grid(True, linestyle=':')
axs[1].set_xticks(p_values)
axs[1].legend()

plt.tight_layout()
plt.show()


## 2D "lambda_tpv" (Regularization Parameter) Evaluation Over One Image
In this cell we evaluate the SSIM and PSNR metrics of one image with "p" fixed to find the best value of the parameter "lambda_tpv"

In [ ]:
import matplotlib.cm as cm

# Calculate the baseline starting metrics comparing the degraded image with the ground truth
psnr_start, ssim_start = compute_metrics(y_noisy, x_clean)

# Generate an array of values for the 'lambda_tpv' parameter (regularization weight)
# We test 6 values uniformly spaced between 1e-6 and 1e-5
# Uncomment the list below if you want to explore non-linear orders of magnitude manually

lambda_values = np.linspace(0.00001, 0.00009, 10)

# Round the lambda values to 6 decimal places to avoid Python's floating-point precision artifacts
for i in range(len(lambda_values)):
  lambda_values[i] = round(lambda_values[i], 6)

# Dictionary to store the optimization results for each tested lambda
results_lambda = {}

print(f"Starting Tuning for the 'lambda_tpv' parameter on the image with noise level {noise_key}...\n")

for l_val in lambda_values:
    print(f"--- Optimization with lambda_tpv = {l_val:.6f} ---")

    # Run the TpV algorithm varying the regularization weight and keeping everything else constant
    x_reconstructed, history = deblur_denoise_tpv_chambolle_pock(
        y=y_noisy,
        K=blur,
        x_clean=x_clean,
        lambda_tpv=l_val,  # The parameter being tuned
        p=0.5,             # Keep the 'p' parameter fixed
        num_iters=500,     # Number of iterations for tuning
        lr=0.01,
        device=DEVICE
    )

    # Store the reconstructed image and its learning history for the current lambda
    results_lambda[l_val] = {
        'x_reconstructed': x_reconstructed,
        'history': history
    }


# PLOT 1: METRICS EVOLUTION OVER ITERATIONS

fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Use a continuous colormap (viridis) to smoothly color the lines representing different lambda values
colors = cm.viridis(np.linspace(0, 0.9, len(lambda_values)))

for i, l_val in enumerate(lambda_values):
    history = results_lambda[l_val]['history']
    c = colors[i]
    label = f'λ = {l_val:.6f}'

    # Plot the learning curves for each execution
    axs[0].plot(history['loss'], color=c, label=label)
    axs[1].plot(history['iters'], history['psnr'], color=c, label=label)
    axs[2].plot(history['iters'], history['ssim'], color=c, label=label)

# Setup subplots details (titles, labels, legends, grids)
axs[0].set_title('Total Loss Evolution')
axs[0].set_xlabel('Iterations')
axs[0].set_ylabel('Loss')
axs[0].legend(fontsize='small', loc='upper right')
axs[0].grid(True)

axs[1].axhline(y=psnr_start, color='gray', linestyle='--', label='Degraded Baseline')
axs[1].set_title('PSNR Evolution')
axs[1].set_xlabel('Iterations')
axs[1].set_ylabel('PSNR [dB]')
axs[1].legend(fontsize='small', loc='lower right')
axs[1].grid(True)

axs[2].axhline(y=ssim_start, color='gray', linestyle='--', label='Degraded Baseline')
axs[2].set_title('SSIM Evolution')
axs[2].set_xlabel('Iterations')
axs[2].set_ylabel('SSIM [0-1]')
axs[2].legend(fontsize='small', loc='lower right')
axs[2].grid(True)

plt.tight_layout()
plt.show()


# PLOT 2: FINAL SUMMARY GRAPHS

# Extract the final metric value reached at the last iteration for each tested lambda
final_psnr = [results_lambda[l_val]['history']['psnr'][-1] for l_val in lambda_values]
final_ssim = [results_lambda[l_val]['history']['ssim'][-1] for l_val in lambda_values]

fig, axs = plt.subplots(1, 2, figsize=(15, 5))

# --- Plot Final PSNR vs lambda ---
axs[0].plot(lambda_values, final_psnr, marker='o', markersize=8, color='dodgerblue', linewidth=2)
axs[0].axhline(y=psnr_start, color='gray', linestyle='--', label='Degraded Baseline')
axs[0].set_title(f'Final PSNR vs "lambda_tpv" (Noise: {noise_key})', fontsize=14)
axs[0].set_xlabel('Parameter "lambda_tpv" Value', fontsize=12)
axs[0].set_ylabel('Final PSNR [dB]', fontsize=12)
axs[0].grid(True, linestyle=':', which='both')

# Optional: Set the X-axis to logarithmic scale if testing wide ranges of magnitude
# axs[0].set_xscale('log')

axs[0].set_xticks(lambda_values)
axs[0].set_xticklabels([str(v) for v in lambda_values])
axs[0].legend()

# --- Plot Final SSIM vs lambda ---
axs[1].plot(lambda_values, final_ssim, marker='s', markersize=8, color='forestgreen', linewidth=2)
axs[1].axhline(y=ssim_start, color='gray', linestyle='--', label='Degraded Baseline')
axs[1].set_title(f'Final SSIM vs "lambda_tpv" (Noise: {noise_key})', fontsize=14)
axs[1].set_xlabel('Parameter "lambda_tpv" Value', fontsize=12)
axs[1].set_ylabel('Final SSIM [0-1]', fontsize=12)
axs[1].grid(True, linestyle=':', which='both')

# Optional: Set the X-axis to logarithmic scale
# axs[1].set_xscale('log')

axs[1].set_xticks(lambda_values)
axs[1].set_xticklabels([str(v) for v in lambda_values])
axs[1].legend()

plt.tight_layout()
plt.show()

## 3D "lambda_tpv" (Regularization Parameter) Evaluetion Over Multiple Images
In this cell, we evaluate the SSIM and PSNR metrics for multiple images with a fixed value of “p” in order to identify the optimal value of the “lambda_tpv” parameter. The results of this cell consist of two 3D plots for the two metrics, along with their mean, median, mode, and standard deviation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats

p_val = 0.5

# Define the number of different images to test (e.g., all the images in the test set)
num_images = 50 #len(final_dataset['test'])
image_indices = np.arange(num_images)

# List of lambda_tpv values to explore during the grid search
lambda_values = np.linspace(0.00001, 0.00009, 5)
#lambda_values = [0.00004,0.00006,0.00008,0.0001, 0.0002,0.0003,0.0004,0.0005,0.0006,0.0007,0.0008]
# Round the lambda values to 6 decimal places to avoid Python's floating-point precision artifacts
for i in range(len(lambda_values)):
  lambda_values[i] = round(lambda_values[i], 6)

# Create a 2D meshgrid for 3D surface plotting
I_mesh, L_mesh = np.meshgrid(image_indices, lambda_values)

# Arrays to store the MAXIMUM values of PSNR/SSIM achieved during the optimization
Z_psnr = np.zeros_like(I_mesh, dtype=float)
Z_ssim = np.zeros_like(I_mesh, dtype=float)

# NEW ARRAYS: To store the exact iteration at which the maximum metric was achieved
Z_psnr_iters = np.zeros_like(I_mesh, dtype=int)
Z_ssim_iters = np.zeros_like(I_mesh, dtype=int)

print(f"Starting Tuning on {num_images} images with p={p_val} while varying lambda...\n")

for idx in range(num_images):
    # Load the specific image sample from the degraded dataset
    sample = ds_degraded[int(idx)]
    x_clean = pil_to_tensor(sample['x']).unsqueeze(0).to(DEVICE).float()
    y_noisy = pil_to_tensor(sample[noise_key]).unsqueeze(0).to(DEVICE).float()

    print(f"=== Processing Image {idx+1}/{num_images} ===")

    for i, l_val in enumerate(lambda_values):
        print(f"  > Optimizing with Lambda = {l_val:.6f} ...", end=" ")

        # Run the TpV algorithm
        x_reconstructed, history = deblur_denoise_tpv_chambolle_pock(
            y=y_noisy,
            K=blur,
            x_clean=x_clean,
            lambda_tpv=l_val,
            p=p_val,
            num_iters=500,
            lr=0.01,
            device=DEVICE
        )

        # Get the index of the highest PSNR achieved in the entire history
        best_psnr_idx = np.argmax(history['psnr'])
        Z_psnr[i, idx] = history['psnr'][best_psnr_idx]
        Z_psnr_iters[i, idx] = history['iters'][best_psnr_idx]

        # Get the index of the highest SSIM achieved in the entire history
        best_ssim_idx = np.argmax(history['ssim'])
        Z_ssim[i, idx] = history['ssim'][best_ssim_idx]
        Z_ssim_iters[i, idx] = history['iters'][best_ssim_idx]

        print(f"Done! (Max PSNR: {Z_psnr[i, idx]:.4f} dB at iteration {Z_psnr_iters[i, idx]})")

print("\nOptimizations completed. Generating 3D graphs...")


# 3D SURFACE PLOTS

fig = plt.figure(figsize=(18, 7))

# Apply Log10 to the Lambda axis to prevent visual clustering/compression of values
L_log = np.log10(L_mesh)

# --- 3D PSNR GRAPH ---
ax1 = fig.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(I_mesh, L_log, Z_psnr, cmap='viridis', edgecolor='k', alpha=0.85)

ax1.set_title(f'PSNR Surface (p={p_val}, Noise={noise_key})', fontsize=16)
ax1.set_xlabel('Image Index', fontsize=12)
ax1.set_ylabel('Lambda parameter (log scale)', fontsize=12)
ax1.set_zlabel('PSNR [dB]', fontsize=12)

# Optional: Disable X ticks if there are too many images
# ax1.set_xticks(image_indices)
ax1.set_yticks(np.log10(lambda_values))
ax1.set_yticklabels([str(v) for v in lambda_values])

# Display the colorbar mapped specifically to this axis (ax=ax1)
fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=10, pad=0.1, label='PSNR')

# --- 3D SSIM GRAPH ---
ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(I_mesh, L_log, Z_ssim, cmap='plasma', edgecolor='k', alpha=0.85)

ax2.set_title(f'SSIM Surface (p={p_val}, Noise={noise_key})', fontsize=16)
ax2.set_xlabel('Image Index', fontsize=12)
ax2.set_ylabel('Lambda parameter (log scale)', fontsize=12)
ax2.set_zlabel('SSIM [0-1]', fontsize=12)

# Optional: Disable X ticks if there are too many images
# ax2.set_xticks(image_indices)
ax2.set_yticks(np.log10(lambda_values))
ax2.set_yticklabels([str(v) for v in lambda_values])

# Display the colorbar mapped specifically to this axis (ax=ax2)
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10, pad=0.1, label='SSIM')

# Adjust the viewing angle for better 3D perspective
ax1.view_init(elev=20, azim=-60)
ax2.view_init(elev=20, azim=-60)

plt.tight_layout()
plt.show()


# BEST PARAMETERS EXTRACTION AND ITERATIONS


# 1. MEAN-BASED CALCULATION
# Calculate the mean and standard deviation across all images (axis 1) for each lambda
mean_psnr = np.mean(Z_psnr, axis=1)
mean_ssim = np.mean(Z_ssim, axis=1)
std_psnr = np.std(Z_psnr, axis=1)
std_ssim = np.std(Z_ssim, axis=1)

# Find the index of the lambda that produced the highest mean metric
idx_best_mean_psnr = np.argmax(mean_psnr)
idx_best_mean_ssim = np.argmax(mean_ssim)

# Retrieve the actual best lambda value
best_lambda_mean_psnr = lambda_values[idx_best_mean_psnr]
best_lambda_mean_ssim = lambda_values[idx_best_mean_ssim]

# Calculate the AVERAGE ITERATION at which the winning lambda reached its peak
avg_iter_mean_psnr = np.mean(Z_psnr_iters[idx_best_mean_psnr, :])
avg_iter_mean_ssim = np.mean(Z_ssim_iters[idx_best_mean_ssim, :])

# 2. MEDIAN-BASED CALCULATION (Robust to outliers)
median_psnr = np.median(Z_psnr, axis=1)
median_ssim = np.median(Z_ssim, axis=1)

idx_best_median_psnr = np.argmax(median_psnr)
idx_best_median_ssim = np.argmax(median_ssim)

best_lambda_median_psnr = lambda_values[idx_best_median_psnr]
best_lambda_median_ssim = lambda_values[idx_best_median_ssim]

avg_iter_median_psnr = np.mean(Z_psnr_iters[idx_best_median_psnr, :])
avg_iter_median_ssim = np.mean(Z_ssim_iters[idx_best_median_ssim, :])

# 3. MODE-BASED CALCULATION (Which lambda won on the highest number of individual images)
best_idx_per_img_psnr = np.argmax(Z_psnr, axis=0)
best_idx_per_img_ssim = np.argmax(Z_ssim, axis=0)

mode_psnr_idx = stats.mode(best_idx_per_img_psnr, keepdims=False).mode
mode_ssim_idx = stats.mode(best_idx_per_img_ssim, keepdims=False).mode

best_lambda_mode_psnr = lambda_values[mode_psnr_idx]
best_lambda_mode_ssim = lambda_values[mode_ssim_idx]

avg_iter_mode_psnr = np.mean(Z_psnr_iters[mode_psnr_idx, :])
avg_iter_mode_ssim = np.mean(Z_ssim_iters[mode_ssim_idx, :])

# 4. LAMBDA STANDARD DEVIATION (Measures how much the images disagree on the best parameter)
best_lambdas_psnr_array = np.array([lambda_values[i] for i in best_idx_per_img_psnr])
best_lambdas_ssim_array = np.array([lambda_values[i] for i in best_idx_per_img_ssim])

std_lambda_psnr = np.std(best_lambdas_psnr_array)
std_lambda_ssim = np.std(best_lambdas_ssim_array)



# FINAL PRINTING OF GLOBAL RESULTS

print("=== GLOBAL RESULTS ===")

print("\n--- PSNR Analysis ---")
print(f"Best Lambda (MEAN):   {best_lambda_mean_psnr} (Mean PSNR: {np.max(mean_psnr):.2f} ± {std_psnr[idx_best_mean_psnr]:.2f} dB) [Reached at average iteration: {avg_iter_mean_psnr:.0f}]")
print(f"Best Lambda (MEDIAN): {best_lambda_median_psnr} (Median PSNR: {np.max(median_psnr):.2f} dB) [Reached at average iteration: {avg_iter_median_psnr:.0f}]")
print(f"Best Lambda (MODE):   {best_lambda_mode_psnr} (Won on {np.sum(best_idx_per_img_psnr == mode_psnr_idx)}/{num_images} img) [Reached at average iteration: {avg_iter_mode_psnr:.0f}]")
print(f"Standard Deviation of 'Ideal Lambdas' chosen individually by images: ± {std_lambda_psnr:.6f}")

print("\n--- SSIM Analysis ---")
print(f"Best Lambda (MEAN):   {best_lambda_mean_ssim} (Mean SSIM: {np.max(mean_ssim):.6f} ± {std_ssim[idx_best_mean_ssim]:.6f}) [Reached at average iteration: {avg_iter_mean_ssim:.0f}]")
print(f"Best Lambda (MEDIAN): {best_lambda_median_ssim} (Median SSIM: {np.max(median_ssim):.6f}) [Reached at average iteration: {avg_iter_median_ssim:.0f}]")
print(f"Best Lambda (MODE):   {best_lambda_mode_ssim} (Won on {np.sum(best_idx_per_img_ssim == mode_ssim_idx)}/{num_images} img) [Reached at average iteration: {avg_iter_mode_ssim:.0f}]")
print(f"Standard Deviation of 'Ideal Lambdas' chosen individually by images: ± {std_lambda_ssim:.6f}")
